In [10]:
import pandas as pd
import os
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.parse import quote_plus
from rdflib.namespace import RDF, DC, Namespace
import xml.etree.ElementTree as ET
from lxml import etree
import shutil
import zipfile
import ftplib
import io
import csv

In [2]:
# Constants
UNZIP_DIR = "selected_data"
FTP_HOST = "download.europeana.eu"
FTP_PATH = "dataset/XML/"

In [4]:
data_ids = []
# extract the dataset_ids
with open('dataset_ids.txt', 'r') as f:
    data_ids = f.readlines()
    data_ids = [x.strip() for x in data_ids]

data_ids = data_ids[1000:1010]
print(data_ids)

['9200249.zip', '2064129.zip', '15416.zip', '574.zip', '92030.zip', '618.zip', '571.zip', '9200211.zip', '254.zip', '2048605.zip']


In [5]:
# Example processing: save the first element to a UNZIP_DIR as an xml file
if not os.path.exists(UNZIP_DIR):
    os.makedirs(UNZIP_DIR)

In [6]:
def download_file(ftp_host, ftp_path, filename):
    zip_data = io.BytesIO()

    with ftplib.FTP(ftp_host) as ftp:
        ftp.login()  # Login as anonymous
        ftp.cwd(ftp_path)

        ftp.retrbinary(f'RETR {filename}', zip_data.write)
    
    zip_data.seek(0)
    return zip_data

# Function to unzip a file
def unzip_file(zip_data):
    extracted_files = []  # List to store file content

    with zipfile.ZipFile(zip_data, 'r') as zip_ref:
        for file_info in zip_ref.infolist():
            with zip_ref.open(file_info) as file:
                file_content = file.read()
                extracted_files.append(file_content)

    return extracted_files

# Function to download and process a ZIP file
def download_and_process_zip(filename):
    try:
        print(f"Starting download and processing for {filename}...")
        
        # Download the ZIP file into memory
        zip_data = download_file(FTP_HOST, FTP_PATH, filename)
        
        # Unzip and process the file
        extracted_files = unzip_file(zip_data)
        print(f"deleting zip file {filename}")
        del zip_data 

        with open(os.path.join(UNZIP_DIR, filename.replace('.zip', '.xml')), 'wb') as xml_file:
            xml_file.write(extracted_files[0])

        print(f"Finished processing {filename}")
        del extracted_files

        try: 
            print(extracted_files[0])
        except Exception as e:
            print(f"Error processing {filename}: {e}")

    except Exception as e:
        print(f"Error processing {filename}: {e}")

In [12]:
# Use ThreadPoolExecutor to download and process files in parallel
with ThreadPoolExecutor(max_workers=10) as executor:
    futures = [executor.submit(download_and_process_zip, filename) for filename in data_ids]

    # Use tqdm to show progress as futures are completed
    for future in tqdm(as_completed(futures), total=len(futures), desc="Processing ZIP files"):
        # This will raise exceptions if any occurred during processing
        future.result()

Starting download and processing for 9200249.zip...
Starting download and processing for 2064129.zip...
Starting download and processing for 15416.zip...
Starting download and processing for 574.zip...
Starting download and processing for 92030.zip...
Starting download and processing for 618.zip...
Starting download and processing for 571.zip...
Starting download and processing for 9200211.zip...
Starting download and processing for 254.zip...
Starting download and processing for 2048605.zip...


Processing ZIP files:   0%|          | 0/10 [00:00<?, ?it/s]

Processing ZIP files:  10%|█         | 1/10 [00:01<00:09,  1.04s/it]

deleting zip file 15416.zip
Error processing 15416.zip: [Errno 2] No such file or directory: 'unzipped/15416.xml'
deleting zip file 9200249.zip
Error processing 9200249.zip: [Errno 2] No such file or directory: 'unzipped/9200249.xml'
deleting zip file 254.zip
Error processing 254.zip: [Errno 2] No such file or directory: 'unzipped/254.xml'


Processing ZIP files: 100%|██████████| 10/10 [00:01<00:00,  6.80it/s]

deleting zip file 92030.zip
Error processing 92030.zip: [Errno 2] No such file or directory: 'unzipped/92030.xml'
deleting zip file 574.zip
Error processing 574.zip: [Errno 2] No such file or directory: 'unzipped/574.xml'
deleting zip file 618.zip
Error processing 618.zip: [Errno 2] No such file or directory: 'unzipped/618.xml'
deleting zip file 9200211.zip
Error processing 9200211.zip: [Errno 2] No such file or directory: 'unzipped/9200211.xml'
deleting zip file 571.zip
Error processing 571.zip: [Errno 2] No such file or directory: 'unzipped/571.xml'
deleting zip file 2048605.zip
Error processing 2048605.zip: [Errno 2] No such file or directory: 'unzipped/2048605.xml'
deleting zip file 2064129.zip
Error processing 2064129.zip: [Errno 2] No such file or directory: 'unzipped/2064129.xml'
